# Amazon Bestselling Books — Process Phase

Standardize column names, canonicalize the one near-duplicate title found in Prepare, and compute
`times_on_list` — the central derived variable for this analysis — then save the cleaned,
row-level dataset. Source: `data/raw/bestsellers_with_categories.csv` (550 rows, 7 columns).

## Step 1 — Load raw data

In [1]:
import pandas as pd
import os

RAW = "../data/raw"
OUT_DIR = "../data/processed"
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(os.path.join(RAW, "bestsellers_with_categories.csv"))
print(f"Loaded: {df.shape}")

Loaded: (550, 7)


## Step 2 — Standardize column names to `snake_case`

In [2]:
df = df.rename(columns={
    "Name": "name", "Author": "author", "User Rating": "user_rating",
    "Reviews": "reviews", "Price": "price", "Year": "year", "Genre": "genre",
})
print(f"Columns after rename: {list(df.columns)}")

Columns after rename: ['name', 'author', 'user_rating', 'reviews', 'price', 'year', 'genre']


## Step 3 — Canonicalize the near-duplicate title

Found in Prepare: *"The 5 Love Languages: The Secret to Love **That** Lasts"* (2010–2014) and
*"...to Love **that** Lasts"* (2015–2019) are the same book, split by a capitalization
inconsistency. Left as two raw spellings, this would undercount a real 10-year repeat bestseller
as two separate 5-year ones. Canonicalized to a single spelling before computing `times_on_list`.

In [3]:
before = df["name"].nunique()
mask = df["name"].str.lower() == "the 5 love languages: the secret to love that lasts"
variants = df.loc[mask, "name"].unique()
print(f"Variants found for the near-duplicate title: {list(variants)}")
df.loc[mask, "name"] = "The 5 Love Languages: The Secret to Love That Lasts"
after = df["name"].nunique()
print(f"Distinct titles before canonicalization: {before}, after: {after}")

Variants found for the near-duplicate title: ['The 5 Love Languages: The Secret to Love That Lasts', 'The 5 Love Languages: The Secret to Love that Lasts']
Distinct titles before canonicalization: 351, after: 350


## Step 4 — Compute `times_on_list`

The central derived variable for the business question: how many distinct years did each
(canonicalized) title appear in the top-50 list? Computed via `groupby(...).nunique()` and merged
back onto every row so each row-level record carries its title's overall repeat count.

In [4]:
times_on_list = df.groupby("name")["year"].nunique().rename("times_on_list")
df = df.merge(times_on_list, on="name", how="left")
print(f"times_on_list computed. Max: {df['times_on_list'].max()}, "
      f"titles with times_on_list > 1: {(times_on_list > 1).sum()}")
print("Top 5 by times_on_list:")
top5 = df[["name", "times_on_list"]].drop_duplicates().sort_values("times_on_list", ascending=False).head(5)
print(top5.to_string(index=False))

times_on_list computed. Max: 10, titles with times_on_list > 1: 95
Top 5 by times_on_list:
                                                                     name  times_on_list
Publication Manual of the American Psychological Association, 6th Edition             10
                      The 5 Love Languages: The Secret to Love That Lasts             10
                                                      StrengthsFinder 2.0              9
                                                Oh, the Places You'll Go!              8
                                              The Very Hungry Caterpillar              7


## Step 5 — Re-verify nulls and exact-duplicate rows after cleaning

In [5]:
print(f"Nulls after cleaning: {df.isnull().sum().sum()}")
print(f"Exact duplicate rows after cleaning: {df.duplicated().sum()}")

Nulls after cleaning: 0
Exact duplicate rows after cleaning: 0


## Step 6 — Re-check the `genre` value set

In [6]:
print(f"Genre values: {sorted(df['genre'].unique())}")

Genre values: ['Fiction', 'Non Fiction']


## Step 7 — Save the processed dataset

In [7]:
out_path = os.path.join(OUT_DIR, "bestsellers_clean.parquet")
df.to_parquet(out_path, index=False)
size_kb = os.path.getsize(out_path) / 1024
print(f"Saved {out_path} ({size_kb:.1f} KB), shape={df.shape}")

Saved ../data/processed\bestsellers_clean.parquet (25.7 KB), shape=(550, 8)


## Verification

The DuckDB SQL pipeline ([`sql/01_process_data.sql`](../sql/01_process_data.sql)) reproduces this
pipeline exactly: 550 rows, 350 distinct titles, the same `times_on_list` value for every title
(max 10, 95 titles repeating), and the same genre split (240 Fiction / 310 Non Fiction). No
cross-engine discrepancy was found — both pipelines agree exactly.

## Summary

| Step | Result |
|---|---|
| Standardized column names | 7 columns renamed to `snake_case` |
| Canonicalized near-duplicate title | 351 → 350 distinct titles |
| Computed `times_on_list` | max 10 (tie: APA Publication Manual, The 5 Love Languages), 95 titles repeat |
| Nulls / exact duplicates after cleaning | 0 / 0 |
| Genre values | `Fiction` (240), `Non Fiction` (310) — confirmed clean |
| Output | `data/processed/bestsellers_clean.parquet`, 550 rows, 8 columns |